# Recherche sectorielle EU / US

Ce notebook sert d'interface pour les visualisations et les backtests.

Modifier uniquement les paramètres de la première cellule, puis exécuter les cellules suivantes.


In [ ]:
# Paramètres principaux

MARKET = "EU"            # "EU" ou "US"
PILLAR = "Value"         # Leverage, Margin, Value, Momentum, Growth, Volatility

# None = toutes les variables du pilier.
# Exemple Value EU : ["price_fcf", "ev_ebitda"]
VARIABLES = None

# Piliers affichés dans la heatmap générale.
# None = tous les piliers.
PILIERS_PLOT = None

# None = dernière date disponible.
DATE = None

# Paramètres du backtest.
TOP_N = 3
START = None             # Exemple : "2015-01-31"
END = None
INCLURE_PILIER = True

# Activation des blocs.
PLOT_PILIERS = True
PLOT_VARIABLES = True
PLOT_RANG_GLOBAL = True
RUN_BACKTEST = True
BACKTEST_PLOT = True

# Séries affichées dans le graphique du backtest.
# Choix possibles : "top", "worst", "long_short", "benchmark"
PLOT_SERIES = ["top", "benchmark"]

# Sauvegarde CSV du backtest.
SAVE_BACKTEST = False
DOSSIER_BACKTEST = "output/backtests"

# Utiliser None pour prendre les fichiers configurés.
FICHIER_EXCEL = None
FICHIER_MACRO = None


In [ ]:
import pandas as pd
from IPython.display import display

from research_secto import RechercheSectorielle

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## Chargement du modèle

Le modèle n'est calculé qu'une seule fois. Les cellules suivantes réutilisent les résultats en mémoire.


In [ ]:
recherche = RechercheSectorielle(
    marche=MARKET,
    fichier_excel=FICHIER_EXCEL,
    fichier_macro=FICHIER_MACRO,
)

recherche.charger()

print("Marché :", recherche.marche)
print("Variables disponibles :")
display(recherche.variables_disponibles())


## Plot des piliers


In [ ]:
if PLOT_PILIERS:
    fig_piliers, ax_piliers = recherche.plot_piliers(
        date=DATE,
        piliers=PILIERS_PLOT,
    )
    display(fig_piliers)


## Plot des variables sélectionnées


In [ ]:
if PLOT_VARIABLES:
    fig_variables, ax_variables = recherche.plot_variables(
        pilier=PILLAR,
        variables=VARIABLES,
        date=DATE,
    )
    display(fig_variables)


## Rang global


In [ ]:
if PLOT_RANG_GLOBAL:
    fig_rang, ax_rang = recherche.plot_rang_global(
        date=DATE,
    )
    display(fig_rang)


## Backtest


In [ ]:
resume = None
backtests = None
fig_backtest = None

if RUN_BACKTEST:
    resume, backtests, fig_backtest, ax_backtest = recherche.backtest(
        pilier=PILLAR,
        variables=VARIABLES,
        inclure_pilier=INCLURE_PILIER,
        top_n=TOP_N,
        start=START,
        end=END,
        avec_plot=BACKTEST_PLOT,
        series_plot=PLOT_SERIES,
    )

    display(resume)

    if fig_backtest is not None:
        display(fig_backtest)


## Détail mensuel

Choisir une variable pour inspecter les secteurs Top / Worst mois par mois.


In [ ]:
if RUN_BACKTEST and backtests:
    nom = next(iter(backtests))
    print("Variable affichée :", nom)
    display(backtests[nom].tail(24))


## Sauvegarde optionnelle


In [ ]:
if SAVE_BACKTEST and RUN_BACKTEST:
    recherche.sauvegarder_backtest(
        resume=resume,
        backtests=backtests,
        pilier=PILLAR,
        dossier=DOSSIER_BACKTEST,
    )
    print("Backtest sauvegardé dans :", DOSSIER_BACKTEST)
